[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zshakespear/bd-spectra-onramp/blob/main/notebooks/03_band_atlas.ipynb)

# 3 · The band atlas

*Brown Dwarf Spectra On-Ramp · about 35 minutes*

**By the end of this notebook you will be able to:**
- name the absorbers that shape a brown dwarf spectrum between 0.75 and 5 µm, and where they act
- read plots with the absorption bands shaded
- define a band, a window, a band's core and wings, and collision-induced absorption
- explain two traps: overlapping bands, and band wings reaching into windows
- explain why band depth tracks absorption, not abundance

> **First time here?** Do *File → Save a copy in Drive* before you change anything (notebook 0 explains why).
> Then run the **Setup** cell below. Run cells in order, top to bottom, with **Shift + Enter**.
> **Moving between notebooks:** use the links inside the notebooks (such as **Next** at the bottom). They always open
> the original notebook, so they work from your copy too — then save a copy of the new notebook before you edit it.

In [ ]:
# ▶ SETUP - run this cell first. It is safe to run more than once.
REPO_URL = "https://github.com/zshakespear/bd-spectra-onramp.git"
import os, subprocess, sys
from pathlib import Path

if "google.colab" in sys.modules:                       # running in Google Colab
    ROOT = Path("/content/bd-spectra-onramp")
    if not (ROOT / "bdtools").exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(ROOT)], check=True)
    else:
        subprocess.run(["git", "-C", str(ROOT), "pull", "--ff-only", "--quiet"], check=False)
else:                                                   # running in Jupyter on your own computer
    ROOT = Path.cwd()
    while not (ROOT / "bdtools").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import matplotlib.pyplot as plt
import bdtools as bd
bd.setup_notebook()

## 1. The atlas

The band positions live in one instructor-editable file, `data/bands.yaml`, so every plot and the
field guide use the same ranges and colours. Each row says where its numbers come from.
`status` tells you where the row comes from: **spec** = stated in Rustamkulov et al. (2026, Sec. 3);
**checked** = range from the band table of Cushing et al. (2005); **uncertain** = listed by Cushing et al. (2005),
but whether SPHEREx can separate it at all is unclear.

In [ ]:
import pandas as pd
rows = []
for name, ab in bd.load_bands()["absorbers"].items():
    for b in ab["bands"]:
        rows.append({"absorber": name, "range (um)": f"{b['range_um'][0]:.2f}-{b['range_um'][1]:.2f}",
                     "types": b["types"], "status": b["status"], "overlaps": b.get("overlaps", ""),
                     "source": b["source"]})
pd.set_option("display.max_colwidth", 70)
pd.DataFrame(rows)

**NH<sub>3</sub>** is in the list for completeness: it is unlikely to be detectable with SPHEREx
(Rustamkulov et al. 2026), so don't go looking for it. Uncertain rows are hidden from plots unless you ask.

### Words we will use
* A **band** is a range of wavelengths where a molecule absorbs strongly. It shows up as a **trough** in the spectrum.
* A **window** is a range *between* bands where little absorbs. There we see deep, hot layers, so the spectrum
  **peaks** — the Y, J, H, K and 4 µm peaks are all windows.
* **H<sub>2</sub> collision-induced absorption (CIA).** A single H<sub>2</sub> molecule barely absorbs infrared light. But
  while two molecules (H<sub>2</sub>–H<sub>2</sub> or H<sub>2</sub>–He) are colliding, the pair can absorb. Because it needs
  collisions, CIA is strongest where the gas is densest (deep down, and in high-gravity objects). It is broad and
  smooth, with no sharp edges, spread over roughly 1.7–3 µm.

In [ ]:
# ✏️ SETTINGS YOU CAN CHANGE
SPECTRAL_TYPE = "L8"
SHOW_UNCERTAIN = False

In [ ]:
spec = bd.load_reference(SPECTRAL_TYPE)
fig, ax = bd.plot_spectrum(spec, log=True, include_uncertain=SHOW_UNCERTAIN,
                           title=f"{spec.name} with its absorption bands shaded")

## 2. Footprints: remove one absorber at a time

The cleanest way to see what an absorber does is to compute the spectrum with and without it.
The curated model grids don't provide spectra with single absorbers removed, so we use the toy —
at SPHEREx-like resolution, so the footprints look the way SPHEREx would see them.

> 🧸 **TOY MODEL** — real band shapes and strengths differ; use this for *where* and *how*, not *how much*.

In [ ]:
r1 = bd.load_toy("remove_one")
labels = list(r1.labels)
full = r1.spherex_flux[labels.index("all")]
others = [lab for lab in labels if lab != "all"]
fig, axes = plt.subplots(1 + len(others), 1, sharex=True, figsize=(9, 9.5),
                         gridspec_kw={"height_ratios": [2.2] + [0.75] * len(others), "hspace": 0.12})
axes[0].plot(r1.spherex_wave_um, full, "k-o", ms=2, label="all toy absorbers")
for ax, lab in zip(axes[1:], others):
    f = r1.spherex_flux[labels.index(lab)]
    name = lab[3:]
    c = bd.molecule_color(name)
    axes[0].plot(r1.spherex_wave_um, f, color=c, lw=1.2, label=lab)
    kept = full / f                      # fraction of the light that gets through this absorber
    ax.fill_between(r1.spherex_wave_um, kept, 1, color=c, alpha=0.5, lw=0)
    ax.plot(r1.spherex_wave_um, kept, color=c, lw=1.2)
    ax.set_ylim(0, 1.05); ax.set_yticks([0, 1])
    ax.text(1.01, 0.5, name.replace("_", " "), transform=ax.transAxes, color=c, fontweight="bold", va="center")
axes[0].set_yscale("log"); axes[0].legend(ncol=3, fontsize=8); axes[0].set_ylabel(r"$F_\lambda$ (toy)")
axes[len(others) // 2 + 1].set_ylabel("fraction of light that gets through\n(filled dip = light this absorber takes out)")
axes[-1].set_xlabel("Wavelength [µm]")
axes[0].set_title("TOY: footprint of each absorber", fontsize=10);

Each lower row is one absorber's **footprint**: the fraction of the light that still gets through when that
absorber is added to all the others. **1 = it takes out nothing; a filled dip = light it takes out**, and a dip all the
way to 0 means it blocks almost all the light there. The top panel shows the same thing as spectra: the coloured
curves are the spectrum with one absorber removed, and the black curve has all of them.

## 3. Trap 1 — overlapping bands

Look at 1.1–1.2 µm and 2.2–2.5 µm in the footprint plot: more than one absorber takes out light there.
When bands overlap, a deeper trough doesn't tell you *which* absorber changed. In real spectra:

* 1.1–1.2 µm: H<sub>2</sub>O and (in T dwarfs) CH<sub>4</sub>;
* 2.2–2.5 µm: CH<sub>4</sub>, CO, and H<sub>2</sub> collision-induced absorption (CIA);
* 3–3.6 µm: CH<sub>4</sub> on the wing of the broad 2.7 µm H<sub>2</sub>O band.

Always check the atlas for *every* absorber in a region before naming one.

## 4. Trap 2 — band wings reach into the windows

### Core, wings and broadening
A band absorbs most strongly at its **core**. Away from the core the absorption gets weaker, but it does not stop
abruptly: these weak tails on either side are the band's **wings**, and they can stretch well into a neighbouring
window. Wings come mainly from **pressure broadening**: molecules that collide often — at high pressure, deep in the
atmosphere — absorb over a wider range of wavelengths, so the band spreads out.

Here is a single made-up band at two pressures. The top panel shows how strongly it absorbs; the bottom panel shows
what fraction of the light gets through.

> 🧸 **TOY** — one invented band, to show the shapes. Real bands are many overlapping lines.

In [ ]:
lam = np.linspace(1.0, 2.0, 2000)          # microns
centre = 1.40

def toy_band(width, wing=0.01):
    x = (lam - centre) / width
    return np.exp(-0.5 * x**2) + wing / (1 + x**2)     # strong core + weak, wide wings

fig, axes = plt.subplots(2, 1, sharex=True, figsize=(9, 7))
for width, label, c in [(0.03, "low pressure", bd.LINE_COLORS[1]), (0.06, "high pressure (broadened)", bd.LINE_COLORS[2])]:
    tau = 300 * toy_band(width)            # optical depth of the band
    axes[0].plot(lam, tau, color=c, label=label)
    axes[1].plot(lam, np.exp(-tau), color=c, label=label)
for ax in axes:
    ax.axvspan(1.60, 1.75, color="0.85", zorder=0)
axes[0].set_yscale("log"); axes[0].set_ylim(1e-2, 1e3)
axes[0].set_ylabel("absorption strength\n(optical depth τ)"); axes[0].legend(loc="upper left")
axes[1].set_ylabel("fraction of light\nthat gets through"); axes[1].set_xlabel("Wavelength [µm]")
axes[1].text(1.40, 0.06, "core:\nalmost no light\ngets through", ha="center", va="bottom", fontsize=9)
axes[1].annotate("wings: weaker absorption\nspreading out to the sides", (1.52, 0.6), (1.77, 0.3),
                 fontsize=9, arrowprops=dict(arrowstyle="->"))
axes[0].text(1.675, 100, "a neighbouring\nwindow", ha="center", va="center", fontsize=9, color="0.3")
axes[0].set_title("TOY: one absorption band, its core and wings", fontsize=10);

The core is so strong that it lets almost nothing through at either pressure (it is *saturated*, notebook 2). The
wings are what change: at high pressure they reach further and take out part of the light in the neighbouring window.

Back to the footprint plot: the dips are not zero in the windows. H<sub>2</sub>O takes out part of the light at the J and H
peaks too, because the wings of its bands reach into those windows. **The height of a window peak depends on its
neighbours.**

In [ ]:
for target in [1.27, 1.60, 4.00]:
    i = np.argmin(abs(r1.spherex_wave_um - target))
    taken = {lab[3:]: f"{(1 - full[i] / f[i]) * 100:.0f}%" for lab, f in zip(labels, r1.spherex_flux) if lab != "all"}
    print(f"window at {target} um - share of the light each absorber takes out:", taken)

## 5. Trap 3 — depth tracks absorption, not abundance

From notebook 2: band depth grows roughly with the *log* of the absorber amount and saturates. So when two
spectra differ in a band, say "**more absorption** at 3.3 µm", not "more methane".

A second reason to be careful: a deeper band does not even have to come from that molecule. Below, we take the
toy atmosphere (T<sub>eff</sub> = 1000 K) and change it in two different ways:

* **top panel:** make the whole atmosphere cooler (T<sub>eff</sub> = 900 K);
* **bottom panel:** keep the temperature, but triple the CH<sub>4</sub> absorption.

Each curve is the **changed spectrum divided by the original**: 1 = no change, below 1 = fainter than before. The
shading shows the absorption bands, as in every plot. **Predict:** which change makes the spectrum fainter
everywhere, and which only in certain bands?

In [ ]:
tv = bd.load_toy("temp_vs_absorber")
base = tv.spherex_flux[0]
fig, axes = plt.subplots(2, 1, sharex=True, figsize=(9, 7))
for ax, k, what, c in [(axes[0], 1, "cooler atmosphere (1000 K -> 900 K): a GLOBAL change", bd.LINE_COLORS[1]),
                       (axes[1], 2, "3 x more CH4 absorption, same temperature: a LOCAL change", bd.LINE_COLORS[5])]:
    bd.shade_bands(ax)
    ax.plot(tv.spherex_wave_um, tv.spherex_flux[k] / base, "o-", ms=3, color=c)
    ax.axhline(1, color="0.4")
    ax.set_ylim(0, 1.1)
    ax.set_ylabel("changed / original")
    ax.text(0.01, 0.04, what, transform=ax.transAxes, fontsize=10, color=c, fontweight="bold")
bd.add_band_key(axes[0])
axes[1].set_xlabel("Wavelength [µm]")
axes[0].set_title("TOY: two ways to change a spectrum", fontsize=10, pad=24);

The cooler atmosphere (top) is fainter at **every** wavelength — most in the blue, where the Planck function is
steepest (notebook 1) — and the bands change depth too: a **global** change. The extra CH<sub>4</sub> (bottom) changes the
spectrum **only inside the CH<sub>4</sub> bands** (orange shading): a **local** change.

So: a difference confined to one absorber's bands points to that absorber; a difference everywhere points to
temperature or another global parameter (notebook 5). Real objects often have both at once.

## 6. Where the real band data come from (pointers only)

Real models use **line lists** — laboratory and theoretical catalogues of millions to billions of individual
molecular transitions (e.g. the ExoMol and HITRAN/HITEMP databases) — and **radiative-transfer codes** that
compute spectra from them (e.g. PICASO, petitRADTRANS). You will not need them in this course; the model
grids in notebook 5 were built with tools like these.

### Exercise 3.1
In the footprint plot, which toy absorber affects the most wavelengths? Is that also true in the real atlas table?

<details><summary><b>Answer</b> (try first, then click)</summary>

H<sub>2</sub>O: it has bands throughout the near-infrared (0.93, 1.15, 1.4, 1.9, 2.7 µm). In real L, T and Y dwarfs,
water is the dominant absorber, carving the troughs between the Y, J, H, K and 4 µm peaks.

</details>

### Exercise 3.2
Two spectra differ only at 2.2–2.5 µm. List every absorber that could be responsible.

<details><summary><b>Answer</b> (try first, then click)</summary>

CH<sub>4</sub>, CO (2.3 µm overtone), and H<sub>2</sub> CIA — and possibly the wing of the 1.9 µm H<sub>2</sub>O band at the
short end. Gravity and metallicity change CIA, so they belong on the list of explanations too (notebook 5).

</details>

## Summary

- H<sub>2</sub>O shapes the whole near-infrared; CH<sub>4</sub> appears at 3–3.6 µm after L4 and defines T dwarfs; CO and CO<sub>2</sub> make notches beyond 4.15 µm; H<sub>2</sub> CIA is broad, ~1.7–3 µm.
- Bands are troughs where a molecule absorbs; windows are the peaks between them, where we see deep.
- Footprints show where an absorber takes out light.
- A band has a strong core and weaker wings; pressure broadening spreads the wings into neighbouring windows.
- Bands overlap, and band wings set the height of window peaks.
- Temperature changes many bands at once; one absorber changes only its own bands.
- Field guide: [Band atlas](https://github.com/zshakespear/bd-spectra-onramp/blob/main/field_guide/band_atlas.md)

**Field guide:** the pages linked above are the short versions to keep open while you work.

## Self-check

Answer these without scrolling up. If one is hard, revisit that section.

1. Name the absorbers that could deepen the trough near 1.15 µm in a T dwarf.
<details><summary><b>Answer</b> (try first, then click)</summary>

H<sub>2</sub>O and CH<sub>4</sub>.

</details>

2. Why can removing H<sub>2</sub>O change the J-peak flux?
<details><summary><b>Answer</b> (try first, then click)</summary>

The wings of neighbouring H<sub>2</sub>O bands reach into the J window.

</details>

3. What is H<sub>2</sub> CIA, and why is it stronger in high-gravity objects?
<details><summary><b>Answer</b> (try first, then click)</summary>

Absorption by pairs of colliding H<sub>2</sub> molecules. It needs collisions, so it is strongest in dense gas — and high gravity means denser gas at the photosphere.

</details>

4. How does a global (temperature) change look different from a local (one absorber) change in a ratio plot?
<details><summary><b>Answer</b> (try first, then click)</summary>

A temperature change moves many bands and the overall slope at once; an absorber change is confined to that absorber's bands.

</details>

5. Why is NH<sub>3</sub> in the atlas but not on the plots?
<details><summary><b>Answer</b> (try first, then click)</summary>

It is unlikely to be detectable with SPHEREx (Rustamkulov et al. 2026).

</details>

**Next:** [Notebook 4 · The spectral sequence](https://colab.research.google.com/github/zshakespear/bd-spectra-onramp/blob/main/notebooks/04_spectral_sequence.ipynb)